# Continuous-time range-only localization with WNOA

This example estimates a 3D trajectory from range measurements to known landmarks. The trajectory is represented by positions and velocities at four knot times, and a white-noise-on-acceleration (WNOA) prior connects consecutive states. Ranges are measured at the first knot and at times between the knots.

The positions use `AugmentedPoint3` and the measurements use `KnownLandmarkSquaredRangeFactorAugmentedPoint3`, both from `gtsam_unstable`. The range factors are written once, on states at their measurement times. From this single set of factors, `interpolateFactorGraph3` builds two problems:

- The **interpolated problem** estimates only the knot states. Each range is evaluated at the WNOA conditional mean of its two neighboring knots, with a fixed measurement noise. We solve it with Levenberg–Marquardt.
- The **full Gaussian-process problem** also estimates a position and velocity at every measurement time, connected to their neighbors by WNOA motion factors. Its squared-range costs act directly on those positions, so it converts exactly to a quadratically constrained quadratic program (QCQP), whose semidefinite relaxation gives a lower bound on its cost.

The two problems are different estimators: the full Gaussian-process problem lets the states at the measurement times deviate from the conditional mean, at a cost set by the WNOA prior.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation, Atlanta, Georgia 30332-0415. All Rights Reserved. Authors: Frank Dellaert, et al. (see THANKS for the full author list). See LICENSE for the license information.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam_unstable/slam/doc/RangeOnlyWnoaAugmentedPoint.ipynb)

In [ ]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass

In [ ]:
import numpy as np
import plotly.graph_objects as go
import gtsam
import gtsam_unstable as gu

## Range measurements and trajectory

We simulate a smooth trajectory observed by four fixed landmarks. Range measurements are taken at the first knot and at two times inside each subsequent knot interval. The knot positions and velocities are the only unknowns; landmark locations remain fixed.

For a landmark at $L_i$ and measured distance $d_i$, the squared-range factor uses

$$
e_i(p(t)) = \|p(t)-L_i\|^2-d_i^2.
$$

If the raw range has standard deviation $\sigma_r$, first-order uncertainty propagation gives $\sigma_{d^2}\approx 2d_i\sigma_r$. We use this value for the one-dimensional factor noise model, since the residual is expressed in squared meters.

In [ ]:
rng = np.random.default_rng(7)

knot_times = np.array([0.0, 1.0, 2.0, 3.0])
measurement_times = np.array([0.0, 0.25, 0.75, 1.25, 1.75, 2.25, 2.75])
landmarks = np.array([
    [0.0, 0.0, 0.0],
    [3.0, 0.0, 0.0],
    [0.0, 3.0, 0.0],
    [0.0, 0.0, 3.0],
])
range_sigma = 0.02  # m
q_psd = np.full(3, 0.2)  # WNOA acceleration power spectral density


def true_position(t):
    t = np.asarray(t)
    return np.stack((
        0.6 + 0.35 * t + 0.06 * np.sin(0.9 * t),
        -0.2 + 0.1 * t - 0.07 * (1 - np.cos(0.7 * t)),
        0.7 - 0.15 * t + 0.06 * np.sin(1.1 * t),
    ), axis=-1)


def true_velocity(t):
    t = np.asarray(t)
    return np.stack((
        0.35 + 0.054 * np.cos(0.9 * t),
        0.1 - 0.049 * np.sin(0.7 * t),
        -0.15 + 0.066 * np.cos(1.1 * t),
    ), axis=-1)


distances = np.linalg.norm(
    true_position(measurement_times)[:, None, :] - landmarks[None, :, :],
    axis=2,
)
measured_ranges = distances + rng.normal(0, range_sigma, distances.shape)

## Build the factor graphs

Each state has a position key and a velocity key. The range factors are created once: on the first knot for the ranges at $t=0$, and on a state at each later measurement time. These measurement-time states only name the times; whether they become optimization variables depends on how the states are partitioned when the graph is built.

For the interpolated problem, `interpolateFactorGraph3` receives the knots as estimated states and the measurement-time states as interpolated states. It adds a `WnoaMotionFactorAugmentedPoint3` between consecutive knots and wraps each range on a measurement-time state in a `WnoaInterpFactorAugmentedPoint3`, which evaluates the range at the interpolated position. The final argument `True` selects the fixed measurement noise model. No variables remain at the intermediate measurement times.

In [ ]:
states = [
    gtsam.StateData(gtsam.symbol("p", i), gtsam.symbol("v", i), float(t))
    for i, t in enumerate(knot_times)
]
# One state per measurement time; the first measurement time is the first knot.
measurement_states = [states[0]] + [
    gtsam.StateData(gtsam.symbol("p", 100 + j), gtsam.symbol("v", 100 + j), float(t))
    for j, t in enumerate(measurement_times) if j > 0
]
knot_set = set(states)
intermediate_set = set(measurement_states[1:])

initial = gtsam.Values()
for state, t in zip(states, knot_times):
    position = true_position(t) + np.array([0.15, -0.10, 0.13])
    velocity = true_velocity(t) + np.array([-0.10, 0.08, -0.06])
    gu.insertAugmentedPoint3(state.pose, gu.AugmentedPoint3(position), initial)
    initial.insert(state.velocity, velocity)


def measurement_graph(keep=lambda index, landmark: True):
    """Squared-range factors on the measurement-time states, for the ranges selected by keep(time_index, landmark_index)."""
    graph = gtsam.NonlinearFactorGraph()
    for index, (state, ranges) in enumerate(zip(measurement_states, measured_ranges)):
        for j, (landmark, distance) in enumerate(zip(landmarks, ranges)):
            if keep(index, j):
                noise = gtsam.noiseModel.Isotropic.Sigma(1, 2 * distance * range_sigma)
                graph.add(gu.KnownLandmarkSquaredRangeFactorAugmentedPoint3(
                    state.pose, landmark, float(distance), noise))
    return graph


def interpolated_graph(measurements):
    """Knots estimated, measurement-time states interpolated with a fixed measurement noise."""
    return gu.interpolateFactorGraph3(measurements, knot_set, intermediate_set, q_psd, True)


measurements = measurement_graph()
graph = interpolated_graph(measurements)
print(f"Knot states: {len(states)}")
print(f"Range measurements: {measurements.size()}")
print(f"Interpolated graph: {graph.size()} factors, {graph.keys().size()} variables")

## Estimate the knot states

Levenberg–Marquardt minimizes the sum of the WNOA motion costs and squared-range measurement costs. Because the problem is nonlinear, the result depends on the initial estimate; the perturbed knot states above provide a nearby starting trajectory.

In [ ]:
result = gtsam.LevenbergMarquardtOptimizer(graph, initial).optimize()


def knot_values(values):
    positions = np.array([
        np.asarray(gu.atAugmentedPoint3(state.pose, values).vector()).reshape(3)
        for state in states
    ])
    velocities = np.array([
        np.asarray(values.atVector(state.velocity)).reshape(3)
        for state in states
    ])
    return positions, velocities


initial_positions, initial_velocities = knot_values(initial)
estimated_positions, estimated_velocities = knot_values(result)

print(f"Initial graph error: {graph.error(initial):.4f}")
print(f"Final graph error:   {graph.error(result):.4f}")
print(f"Knot position RMSE: {np.sqrt(np.mean(np.sum((estimated_positions - true_position(knot_times))**2, axis=1))):.4f} m")

## Trajectory estimate

The estimated knot positions and velocities define a continuous trajectory between the knots. For the Euclidean WNOA model, the conditional mean follows cubic Hermite interpolation. The 3D plot compares that trajectory with ground truth and the initial estimate; the landmarks are shown for spatial reference.

In [ ]:
def interpolate_positions(times, positions, velocities):
    segment = np.clip(
        np.searchsorted(knot_times, times, side="right") - 1,
        0, len(knot_times) - 2,
    )
    dt = knot_times[segment + 1] - knot_times[segment]
    u = (times - knot_times[segment]) / dt
    return (
        (2 * u**3 - 3 * u**2 + 1)[:, None] * positions[segment]
        + (u**3 - 2 * u**2 + u)[:, None] * dt[:, None] * velocities[segment]
        + (-2 * u**3 + 3 * u**2)[:, None] * positions[segment + 1]
        + (u**3 - u**2)[:, None] * dt[:, None] * velocities[segment + 1]
    )


plot_times = np.linspace(knot_times[0], knot_times[-1], 200)
true_path = true_position(plot_times)
initial_path = interpolate_positions(plot_times, initial_positions, initial_velocities)
estimated_path = interpolate_positions(plot_times, estimated_positions, estimated_velocities)

fig = go.Figure()
fig.add_scatter3d(x=true_path[:, 0], y=true_path[:, 1], z=true_path[:, 2], mode="lines", name="Ground truth", line=dict(width=5))
fig.add_scatter3d(x=initial_path[:, 0], y=initial_path[:, 1], z=initial_path[:, 2], mode="lines", name="Initial estimate", line=dict(width=3, dash="dash"))
fig.add_scatter3d(x=estimated_path[:, 0], y=estimated_path[:, 1], z=estimated_path[:, 2], mode="lines", name="WNOA estimate", line=dict(width=5))
fig.add_scatter3d(x=estimated_positions[:, 0], y=estimated_positions[:, 1], z=estimated_positions[:, 2], mode="markers", name="Estimated knots", marker=dict(size=5))
fig.add_scatter3d(x=landmarks[:, 0], y=landmarks[:, 1], z=landmarks[:, 2], mode="markers", name="Known landmarks", marker=dict(size=6, symbol="diamond"))
fig.update_layout(title="Range-only trajectory estimate", scene=dict(xaxis_title="x (m)", yaxis_title="y (m)", zaxis_title="z (m)", aspectmode="data"), margin=dict(l=0, r=0, b=0, t=45))
fig.show()

## Position error

The Euclidean distance between the estimated and true positions measures trajectory accuracy in meters. This plot evaluates the continuous trajectories, not just their knot positions.

In [ ]:
initial_error = np.linalg.norm(initial_path - true_path, axis=1)
estimated_error = np.linalg.norm(estimated_path - true_path, axis=1)

fig = go.Figure()
fig.add_scatter(x=plot_times, y=initial_error, mode="lines", name="Initial estimate", line=dict(dash="dash"))
fig.add_scatter(x=plot_times, y=estimated_error, mode="lines", name="WNOA estimate")
fig.update_layout(title="Trajectory position error", xaxis_title="Time (s)", yaxis_title="Position error (m)")
fig.show()

print(f"Mean position error, initial: {initial_error.mean():.4f} m")
print(f"Mean position error, WNOA:    {estimated_error.mean():.4f} m")

## Full Gaussian-process formulation and its QCQP

A squared-range residual is quadratic in the position, so each position can be written with the lifted vector $X=[h, x, u]$, where $x$ holds the coordinates, $u=x^\top x$, and $h=1$. The lift is exact when $x^\top x = hu$ and $h^2=1$. In this form the squared-range cost is quadratic in $X$, and the WNOA motion cost is already quadratic in positions and velocities, which stay ordinary 3D vectors. Calling `gtsam.QcqpProblem(graph)` performs this conversion without approximation, and every constraint it creates involves a single variable.

The interpolated problem has no such exact form. Its ranges are evaluated at interpolated positions, which are linear in four knot variables, so the squared-range costs are quartic in the knot states. Writing them as a QCQP would need constraints that link several variables, which GTSAM's semidefinite relaxation does not support.

The full Gaussian-process problem avoids this. Calling `interpolateFactorGraph3` with every distinct state estimated and none interpolated keeps each range on its own measurement-time state and connects all consecutive states, knots and measurement times alike, by WNOA motion factors. Because the WNOA prior is Markov, the motion factors along an interval add up to the knot-to-knot prior plus a penalty for deviating from the conditional mean, weighted by the conditional covariance, so the interval prior is not counted twice. With every measurement-time state at its conditional mean, the full Gaussian-process cost equals the interpolated cost; otherwise those states may deviate from the conditional mean, so the full problem's optimum is at most the interpolated optimum.

The quadratic constraints allow $h=\pm 1$ separately for every point, which would flip the sign of the coordinates in the motion costs that couple neighboring points, so we also fix $h=1$ on every position.

The semidefinite relaxation replaces the rank-one matrix $[1;y][1;y]^\top$, built from all lifted variables $y$, with a positive semidefinite matrix whose first entry is 1. Its optimal cost is a lower bound on the QCQP cost. If a feasible point of the QCQP attains that lower bound, the point is a global minimizer of the full Gaussian-process problem and the relaxation is tight for this instance. A numerical solver only reaches the bound up to its tolerance, so the bound is meaningful only when the solver reports an optimal status, and a point that matches it to within that tolerance is optimal only to the same accuracy. If the gap is clearly nonzero, either the relaxation is not tight or the solver did not converge; the bound then only limits how far from optimal any solution can be. Solving the relaxation requires MOSEK, which GTSAM uses through `MosekMonolithicSDP` when GTSAM is configured with `GTSAM_WITH_MOSEK=ON`.

## Sparse intermediate-time measurements

To make the problem sparse, we keep all four ranges at the first knot but only two ranges, to alternating landmarks, at each intermediate time. This leaves 16 of the 28 measurements. The same sparse measurement factors give both problems, and Levenberg–Marquardt solves each of them. The full Gaussian-process problem starts from the interpolated solution, with each measurement-time state at its conditional mean from `updateInterpValues3`, where its cost equals the interpolated cost.

In [ ]:
def sparse_ranges(index, landmark):
    return index == 0 or landmark in (index % 4, (index + 1) % 4)


sparse_measurements = measurement_graph(sparse_ranges)
sparse_graph = interpolated_graph(sparse_measurements)
sparse_result = gtsam.LevenbergMarquardtOptimizer(sparse_graph, initial).optimize()

# Full Gaussian-process problem: every distinct state estimated, none interpolated.
all_states = knot_set | intermediate_set
full_gp_graph = gu.interpolateFactorGraph3(sparse_measurements, all_states, set(), q_psd)
full_gp_initial = gu.updateInterpValues3(sparse_graph, sparse_result, knot_set, intermediate_set, q_psd)
full_gp_result = gtsam.LevenbergMarquardtOptimizer(full_gp_graph, full_gp_initial).optimize()

problem = gtsam.QcqpProblem(full_gp_graph)
fix_h = np.array([[1.0, 0.0, 0.0, 0.0, 0.0]])
for state in all_states:
    problem.addConstraint(gtsam.LinearConstraint.Equal(
        gtsam.JacobianFactor(state.pose, fix_h, np.ones(1), gtsam.noiseModel.Unit.Create(1))))

knot_difference = np.abs(knot_values(full_gp_result)[0] - knot_values(sparse_result)[0]).max()
print(f"Sparse measurements: {sparse_measurements.size()} range factors")
print(f"Interpolated problem, Levenberg–Marquardt cost:      {sparse_graph.error(sparse_result):.6f}")
print(f"Full GP problem at the interpolated solution:          {full_gp_graph.error(full_gp_initial):.6f}")
print(f"Full GP problem, Levenberg–Marquardt cost:            {full_gp_graph.error(full_gp_result):.6f}")
print(f"Largest knot coordinate difference between the two estimates: {knot_difference:.4f} m")
print(f"Full GP graph: {full_gp_graph.size()} factors, {full_gp_graph.keys().size()} variables")
print(f"QCQP (cost, equality, inequality) dimensions: {problem.dim()}")

## Solve the relaxation

MOSEK solves the relaxation of the full Gaussian-process QCQP and returns one lifted column per variable. The relaxation's objective is a lower bound on the full Gaussian-process cost. Any set of states is feasible for the nonlinear problem, so its nonlinear cost is an upper bound. We evaluate that cost at the states read from the SDP solution and also keep the Levenberg–Marquardt cost. A gap near zero, within the solver's accuracy, means the best upper bound is optimal to that accuracy and the relaxation is tight for this instance. A clearly positive gap means the relaxation is not tight or the solver did not converge. Because the solver stops at a finite tolerance, the gap can be slightly negative; a magnitude near that tolerance counts as zero, and a clearly negative gap indicates a problem with the solve or the model. The bound applies to the full Gaussian-process problem, not to the interpolated problem. The cell below skips the solve with an explanation if this GTSAM build has no MOSEK support or no usable license.

In [ ]:
sdp = None
if not hasattr(gtsam, "MosekMonolithicSDP"):
    print("This GTSAM build has no MOSEK support (configure GTSAM with GTSAM_WITH_MOSEK=ON); skipping the SDP.")
else:
    try:
        sdp = gtsam.MosekMonolithicSDP(problem)
        if not sdp.solve():
            raise RuntimeError("the solver did not return a readable solution")
        status = sdp.problemStatus()
        if status != "ProblemStatus::PrimalAndDualFeasible":
            raise RuntimeError(f"cannot interpret the result as an optimum: {status}")
    except RuntimeError as error:
        print(f"MOSEK could not solve the relaxation: {error}")
        sdp = None

In [ ]:
def knot_errors(values):
    positions, _ = knot_values(values)
    return np.linalg.norm(positions - true_position(knot_times), axis=1)


if sdp is not None:
    qcqp_values = sdp.qcqpValues()
    sdp_values = gtsam.Values()  # all full GP states in the original (unlifted) variables
    for state in all_states:
        sdp_values.insert(state.velocity, qcqp_values.atMatrix(state.velocity)[:, 0])
        gu.insertAugmentedPoint3(
            state.pose, gu.fromQcqpValue3(qcqp_values.atMatrix(state.pose)), sdp_values)

    lower_bound = sdp.objectiveValue()
    sdp_cost = full_gp_graph.error(sdp_values)
    lm_cost = full_gp_graph.error(full_gp_result)
    _, violation, _ = problem.evaluate(qcqp_values)
    print(f"MOSEK problem status:                      {sdp.problemStatus()}")
    print(f"SDP lower bound:                           {lower_bound:.6f}")
    print(f"Full GP cost at the SDP states:            {sdp_cost:.6f}")
    print(f"Full GP cost, Levenberg–Marquardt:         {lm_cost:.6f}")
    print(f"Relative gap to the best upper bound:      {(min(sdp_cost, lm_cost) - lower_bound) / lower_bound:.2e}")
    print(f"Constraint violation of the lifted SDP solution: {violation:.2e}")

    fig = go.Figure()
    for name, values in (("Initial estimate", initial), ("Interpolated, Levenberg–Marquardt", sparse_result),
                         ("Full GP, Levenberg–Marquardt", full_gp_result), ("Full GP, SDP relaxation", sdp_values)):
        fig.add_bar(x=[f"Knot {i}" for i in range(len(states))], y=knot_errors(values), name=name)
    fig.update_layout(title="Knot position error, sparse measurements", yaxis_title="Position error (m)", barmode="group")
    fig.show()

## Rank diagnostics

For each variable, the eigenvalue ratio (EVR) divides the two largest eigenvalues of its block of the relaxed matrix, $[1\ y^\top; y\ Y]$. A very large ratio means the block is numerically rank one, so the relaxation produced an exact lifted point for that variable. The constraint violation printed above measures how far the lifted SDP solution is from satisfying the lift constraints and the interpolation links; a block that is exactly rank one satisfies them, but a small violation does not by itself prove that a block is rank one. These diagnostics describe the solution returned by the solver, not every optimal solution, because an optimal relaxation can have several solutions of different rank.

A position that has no direct range measurement, here every knot except the first, enters the costs only through its coordinates $x$ in the motion terms, so nothing in the objective constrains the second moments involving its $u$ entry. Its ratio can therefore be small even when its cost is unaffected. For such blocks, the gap between the lower bound and the upper bound is the more reliable indicator of tightness than the ratio.

In [ ]:
if sdp is not None:
    names = [gtsam.Symbol(key).string() for key in sdp.orderedKeys()]
    ratios = sdp.variableEVRs()
    fig = go.Figure(go.Bar(x=names, y=ratios))
    fig.update_layout(title="Eigenvalue ratio per variable", yaxis_title="EVR", yaxis_type="log")
    fig.show()
    for name, ratio in zip(names, ratios):
        print(f"{name:>5}: EVR {ratio:.2e}")

The range observations constrain the trajectory at their actual measurement times, even though only the knot states are estimated. The WNOA prior couples neighboring positions and velocities, giving a smooth trajectory between observations. Unlike single-point range localization, this graph estimates **a continuous path** from asynchronous range measurements.